Ridewise -- Data Preprocessing

fixing all data quality issues identified during our Eda prepare clean dataset for feature engineering & Modelling

In [2]:
# Import Libraries
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# configurations 
warnings.filterwarnings("ignore")
%matplotlib inline

sns.set_theme(style="whitegrid", font_scale=1.2)

# define the location of our raw and processed data
DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")

# create directories if they don't exist
os.makedirs(DATA_RAW, exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)


Ridewise_London

- data
     
     -raw/
     
     -processed/

In [10]:

riders = pd.read_csv(os.path.join(DATA_RAW,"riders.csv"), parse_dates=["signup_date"])
drivers = pd.read_csv(os.path.join(DATA_RAW,"drivers.csv"), parse_dates=["signup_date"])
trips = pd.read_csv(os.path.join(DATA_RAW,"trips.csv"), parse_dates=["pickup_time", "dropoff_time"])
sessions = pd.read_csv(os.path.join(DATA_RAW,"sessions.csv"))
promotions = pd.read_csv(os.path.join(DATA_RAW,"promotions.csv"))


Step 1 Cleaning Exercise (Riders Dataset)

In [11]:
riders.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,NaN
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,NaN
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,NaN
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002


 1 - referred by
 
 2 - signup date
 
 3 - churn prob(threshold) 
 
 4 - Age --- (ranges)

In [12]:
# make a copy of the original dataframe to work with

riders_clean = riders.copy()

# step 1a -- fix age(round up the values)
riders_clean["age"] = riders_clean["age"].round().astype(int)


# step 1b-- churn label (converting from continuous prob to binary labels)
#e.g (0.124,0.435)
# e.g (1 & 0)

# step 1b(i) --- create a threshold (0.5-50%)
# if churn probability is>50% = churned
# if churn probability is < 50% = Not churned

riders_clean["churned"]= (riders_clean["churn_prob"]>0.5).astype(int)
riders_clean = riders_clean.drop(columns=["churn_prob"])

#step 1c - fix the referred by returning those that were refered
riders_clean["was_referred"]=riders_clean["referred_by"].notna().astype(int)
riders_clean = riders_clean.drop(columns=["referred_by"])


riders_clean.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churned,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,0,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,1,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,1,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,0,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,0,1


Step 2- Cleaning Exercise (trips)

In [17]:
trips.head()

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 19:27:14+02:05,2024-06-18 19:32:14+02:05,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 09:58:16+02:27,2024-10-05 10:28:16+02:27,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold


In [14]:
# copy the trips dataset
trips_clean = trips.copy()

# step 2a -- fix the data columns (convert to datetime)(drop_off and pickup_time)
trips["pickup_time"] = pd.to_datetime(trips["pickup_time"], errors="coerce")
trips["dropoff_time"] = pd.to_datetime(trips["dropoff_time"], errors="coerce")



trips_clean= trips_clean.dropna(subset=["pickup_time","dropoff_time"])


#print out all attributes
print(f"Trips columns data-types:\n{trips_clean.dtypes}")
trips_clean.head()


Trips columns data-types:
trip_id                                object
user_id                                object
driver_id                              object
fare                                  float64
surge_multiplier                      float64
tip                                   float64
payment_type                           object
pickup_time         datetime64[ns, UTC+02:27]
dropoff_time        datetime64[ns, UTC+02:27]
pickup_lat                            float64
pickup_lng                            float64
dropoff_lat                           float64
dropoff_lng                           float64
weather                                object
city                                   object
loyalty_status                         object
dtype: object


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 05:36:41+02:27,2025-02-17 05:52:41+02:27,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 09:58:16+02:27,2024-10-05 10:28:16+02:27,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold
5,T000005,R05852,D04849,16.98,1.4,0.52,Mobile Money,2024-07-02 18:20:12+02:27,2024-07-02 18:52:12+02:27,-1.370880,37.041389,-1.410399,37.045473,Rainy,Nairobi,Silver
7,T000007,R04890,D02006,25.62,1.7,0.00,Mobile Money,2024-12-06 17:23:11+02:27,2024-12-06 18:06:11+02:27,-1.153775,36.358126,-1.134236,36.392381,Rainy,Nairobi,Gold


In [8]:
# Engineered some new features from the trips dataset

# 1. Trip duration(in minutes)
trips_clean["trip_duration"] = (

(trips_clean ["dropoff_time"] - trips_clean["pickup_time"]).dt.total_seconds()/60)


# 2.  -- # total revenue
trips_clean ["total_revenue"] = trips_clean["fare"] * trips_clean["surge_multiplier"] + trips_clean["tip"].fillna(0)

# 3 step 1c -- hour of the day and day of the week
trips_clean["hour_of_day"] = trips_clean["pickup_time"].dt.hour

# 4 Day of week
trips_clean["day_of_week"] = trips_clean["pickup_time"].dt.day_name()

AttributeError: Can only use .dt accessor with datetimelike values

In [6]:
# Data Quality checks

# Remove trips with non-positive duration
trips_clean = trips_clean[trips_clean["trip_duration"] > 0]

# Fill missing tip values
trips_clean["tip"] = trips_clean["tip"].fillna(0)

# Fill missing weather values
trips_clean["weather"] = trips_clean["weather"].fillna("unknown")

print(f"Rows shape: {trips_clean.shape}")

trips_clean.head()

KeyError: 'trip_duration'

In [5]:
# Step 1a -  Trip duration

trips["pickup_time"] = pd.to_datetime(trips["pickup_time"], errors="coerce")
trips["dropoff_time"] = pd.to_datetime(trips["dropoff_time"], errors="coerce")

trips["trip_duration_min"] = (
    (trips["dropoff_time"] - trips["pickup_time"]).dt.total_seconds()/ 60
)

# step 1b -- total revenue
trips["total_revenue"] = trips["fare"] * trips["surge_multiplier"] + trips["tip"].fillna(0)

# step 1c -- hour of the day and day of the week
trips["hour_of_day"] = trips["pickup_time"].dt.hour
trips["day_of_week"] = trips["pickup_time"].dt.day_name()

In [ ]:
# Step 1a -  Trip duration

trips["trip_duration_min"] = (
    (trips["dropoff_time"] - trips["pickup_time"]).dt.total_seconds() / 60
)

# step 1b -- total revenue
trips["total_revenue"] = trips["fare"] * trips["surge_multiplier"] + trips["tip"].fillna(0)

# step 1c -- hour of the day and day of the week
trips["hour_of_day"] = trips["pickup_time"].dt.hour
trips["day_of_week"] = trips["pickup_time"].dt.day_name()

Step 3 - Cleaning Drivers Dataset

In [ ]:
drivers_clean = drivers.copy()
drivers_clean.head()

,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
0,D00000,3.1,SUV,2025-01-20,2025-01-06 18:23:09.312275,Cairo,0.679555
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773


In [ ]:
drivers_clean["rating"]= drivers["rating"].fillna(drivers_clean["rating"].median())
drivers_clean["acceptance_rate"]= drivers["acceptance_rate"].fillna(drivers_clean["acceptance_rate"].median())


print(drivers_clean.dtypes)
print(drivers_clean.isnull().sum())

driver_id                  object
rating                    float64
vehicle_type               object
signup_date        datetime64[ns]
last_active                object
city                       object
acceptance_rate           float64
dtype: object
driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


Step 4- Cleaning exercise for session dataset

In [ ]:
sessions_clean = sessions.copy()

# Step 4(1) - fix the session_time date format
sessions_clean["session_time"] = (
    pd.to_datetime(
        sessions_clean["session_time"],
        utc=True,
        errors="coerce"
    ).dt.tz_localize(None)
)

# Step 4(2) - drop rows with invalid session_time
sessions_clean = sessions_clean.dropna(subset=["session_time"])

print(f"sessions_clean shape: {sessions_clean.shape}")

sessions_clean.head()

sessions_clean shape: (50000, 8)


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 16:52:06,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 05:05:22,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 21:12:25,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:26:25,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:17:22,4,1,0,Lagos,Bronze


step 5 = Referential Integrity

In [ ]:
# validating riders and drivers dataset 

valid_riders = set(riders_clean["user_id"])
valid_drivers = set(drivers_clean["driver_id"])

# check the trips & session dataset
before_trips = len(trips_clean)
before_sessions = len(sessions_clean)

# filter   check if riders are in trips dataset
trips_clean = trips_clean[trips_clean["user_id"].isin(valid_riders)]
trips_clean = trips_clean[trips_clean["driver_id"].isin(valid_drivers)]
sessions_clean = sessions_clean[sessions_clean["rider_id"].isin(valid_riders)]

# check the descrepancies
print(f"Trips that were dropped : {before_trips-len(trips_clean)}")
print(f"sessions that were dropped : {before_sessions - len(sessions_clean)}")
print("Everything looks good")


Trips that were dropped : 0
sessions that were dropped : 0
Everything looks good


Validate & Save

In [ ]:
print("Row count-raw vs clean")
for name, raw, clean in [
    ("riders", riders, riders_clean),
    ("drivers", drivers, drivers_clean),
    ("trips", trips, trips_clean),
    ("sessions", sessions, sessions_clean) 

]:
    print(f"  {name:<10}:{len(raw):>7,}-> {len(clean):>7,}")



Row count-raw vs clean
  riders    : 10,000->  10,000
  drivers   :  5,000->   5,000
  trips     :200,000-> 200,000
  sessions  : 50,000->  50,000


In [ ]:
files = {

    "riders_clean.csv": riders_clean,
    "drivers_clean.csv": drivers_clean,
    "trips_clean.csv": trips_clean,
    "sessions_clean.csv": sessions_clean

}

for fname, df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print(f"{fname} saved successfully")

riders_clean.csv saved successfully
drivers_clean.csv saved successfully
trips_clean.csv saved successfully
sessions_clean.csv saved successfully
